# LLMs for Text Transformation

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Translation

ChatGPT is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.

In [3]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)

Hola, me gustaría ordenar una licuadora.


In [4]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

This is French.


In [5]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

French: Je veux commander un ballon de basket
Spanish: Quiero ordenar un balón de baloncesto
English: I want to order a basketball


In [6]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formal: ¿Le gustaría ordenar una almohada?
Informal: ¿Te gustaría ordenar una almohada?


### Universal Translator
Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!

In [7]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [8]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (French): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다." 

Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."

Korean: "내 모니터에는 빛나지 않는 픽셀이 있습니다." 

Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다 

Original message (This is Polish.): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요 

Original message (This is Chinese.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다 



## Tone Transformation
Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [9]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

Dear Sir/Madam,

I am writing to bring to your attention the specifications of the standing lamp. 

Sincerely,
Joe


## Format Conversion
ChatGPT can translate between formats. The prompt should describe the input and output formats.

In [10]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

<html>
<head>
  <title>Restaurant Employees</title>
</head>
<body>
  <table>
    <tr>
      <th>Name</th>
      <th>Email</th>
    </tr>
    <tr>
      <td>Shyam</td>
      <td>shyamjaiswal@gmail.com</td>
    </tr>
    <tr>
      <td>Bob</td>
      <td>bob32@gmail.com</td>
    </tr>
    <tr>
      <td>Jai</td>
      <td>jai87@gmail.com</td>
    </tr>
  </table>
</body>
</html>


In [11]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

Name,Email
Shyam,shyamjaiswal@gmail.com
Bob,bob32@gmail.com
Jai,jai87@gmail.com


## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response. 

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.

In [12]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies has a ball.
No errors found
No errors found.
Their goes my freedom. There going to bring they’re suitcases.

No errors found.

Rewritten:
Their goes my freedom. There going to bring their suitcases.
You're going to need your notebook.
No errors found.
No errors found


In [13]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)

I got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it's super soft and cute. One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.


In [14]:
# ! pip3 install redlines

In [15]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

ModuleNotFoundError: No module named 'redlines'

In [ ]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

I purchased this adorable panda plush as a birthday gift for my daughter, as she kept borrowing mine from my room. It's not just for kids - even adults can appreciate the charm of pandas. The plush is incredibly soft and cute, and my daughter carries it everywhere with her. However, I did notice that one of the ears is slightly lower than the other, which seems like a manufacturing flaw rather than intentional design. While I found the plush to be a bit smaller than expected given the price, I believe there are larger options available for the same cost. Despite this, the plush arrived a day earlier than anticipated, allowing me to enjoy it myself before passing it on to my daughter. Overall, it's a delightful gift that brings joy to both children and adults alike.

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## English text (with intentional mistakes):

In the early 1900s, before refrigerators were common in homes, food spoiled very quickly. People often used sauces to hide the taste of meals that were already going bad. One of the most popular creations from that time was ketchup, a tomato-based sauce that helped mask unpleasant flavors. It wasnt invented for gourmet reasons, but rather as a practical solution to a real problem: keeping food edible for longer even when it was not fresh anymore.


### Exercise 1 — Translation (Pseudocode)

1. Define the original text about the invention of ketchup.
2. Create a prompt asking the LLM to translate the text into:
   - French
   - Spanish
   - Japanese
3. Send the prompt to the model and store the response.
4. Print the translated output.


In [16]:
text = """
In the early 1900s, before refrigerators were common in homes, food spoiled very quickly. 
People often used sauces to hide the taste of meals that were already going bad. 
One of the most popular creations from that time was ketchup, a tomato-based sauce that helped mask unpleasant flavors. 
It wasnt invented for gourmet reasons, but rather as a practical solution to a real problem: 
keeping food edible for longer even when it was not fresh anymore.
"""

prompt = f"""
Translate the following text into French, Spanish, and Japanese.
Text: ```{text}```
"""

response = get_completion(prompt)
print("=== TRANSLATION OUTPUT ===")
print(response)


=== TRANSLATION OUTPUT ===
French:
Au début des années 1900, avant que les réfrigérateurs ne soient courants dans les foyers, la nourriture se gâtait très rapidement. Les gens utilisaient souvent des sauces pour masquer le goût des repas qui étaient déjà en train de tourner. L'une des créations les plus populaires de cette époque était le ketchup, une sauce à base de tomate qui aidait à masquer les saveurs désagréables. Il n'a pas été inventé pour des raisons gastronomiques, mais plutôt comme une solution pratique à un problème réel : conserver les aliments comestibles plus longtemps même s'ils n'étaient plus frais.

Spanish:
A principios de 1900, antes de que los refrigeradores fueran comunes en los hogares, los alimentos se echaban a perder muy rápidamente. La gente solía usar salsas para ocultar el sabor de las comidas que ya estaban en mal estado. Una de las creaciones más populares de esa época fue el ketchup, una salsa a base de tomate que ayudaba a enmascarar sabores desagradabl

### Exercise 2 — Spellcheck & Grammar Check (Pseudocode)
1. Take each translated version (French, Spanish, Japanese).
2. For each language:
   a. Create a prompt instructing the LLM to proofread and correct the text.
   b. Ask the model to output only the corrected version.
3. Print the corrected versions for all languages.



In [20]:
translations = {
    "French": """Au début des années 1900, avant que les réfrigérateurs ne soient courants dans les foyers, la nourriture se gâtait très rapidement. Les gens utilisaient souvent des sauces pour masquer le goût des repas qui étaient déjà en train de tourner. L'une des créations les plus populaires de cette époque était le ketchup, une sauce à base de tomate qui aidait à masquer les saveurs désagréables. Il n'a pas été inventé pour des raisons gastronomiques, mais plutôt comme une solution pratique à un problème réel : conserver les aliments comestibles plus longtemps même s'ils n'étaient plus frais.""",

    "Spanish": """A principios de 1900, antes de que los refrigeradores fueran comunes en los hogares, los alimentos se echaban a perder muy rápidamente. La gente solía usar salsas para ocultar el sabor de las comidas que ya estaban en mal estado. Una de las creaciones más populares de esa época fue el ketchup, una salsa a base de tomate que ayudaba a enmascarar sabores desagradables. No fue inventado por razones gourmet, sino más bien como una solución práctica a un problema real: mantener los alimentos comestibles por más tiempo incluso cuando ya no estaban frescos.""",

    "Japanese": """1900年代初頭、冷蔵庫が家庭に一般的ではなかった時代、食べ物は非常に早く腐敗していました。人々はよく、すでに腐っている食事の味を隠すためにソースを使っていました。その時代から最も人気のある創作の1つがケチャップで、トマトベースのソースで、不快な味を隠すのに役立ちました。それは美食のために発明されたのではなく、むしろ実際の問題に対する実用的な解決策としてでした。新鮮でなくなっても食べ物を長持ちさせることが目的でした。"""
}

for lang, t in translations.items():
    prompt = f"""
    You are a proofreading assistant. 
    1. Proofread and correct the following {lang} text.
    2. Then compare the corrected version with the original.
    3. Count how many corrections you made (grammar, spelling, punctuation, or wording).
    4. Output the result in this format:

    Corrections made: <number>
    Corrected text: <corrected version>

    Text: ```{t}```
    """
    response = get_completion(prompt)
    print(f"=== {lang.upper()} CORRECTION REPORT ===")
    print(response)
    print("\n")


=== FRENCH CORRECTION REPORT ===
Corrections made: 2
Corrected text: Au début des années 1900, avant que les réfrigérateurs ne soient courants dans les foyers, la nourriture se gâtait très rapidement. Les gens utilisaient souvent des sauces pour masquer le goût des repas qui étaient déjà en train de tourner. L'une des créations les plus populaires de cette époque était le ketchup, une sauce à base de tomate qui aidait à masquer les saveurs désagréables. Il n'a pas été inventé pour des raisons gastronomiques, mais plutôt comme une solution pratique à un problème réel : conserver les aliments comestibles plus longtemps même s'ils n'étaient plus frais.


=== SPANISH CORRECTION REPORT ===
Corrections made: 2
Corrected text: A principios de 1900, antes de que los refrigeradores fueran comunes en los hogares, los alimentos se echaban a perder muy rápidamente. La gente solía usar salsas para ocultar el sabor de las comidas que ya estaban en mal estado. Una de las creaciones más populares de e

### Exercise 3 — Tone Transformation (Pseudocode)
1. Use the corrected English text from the original paragraph.
2. Create a prompt asking the LLM to rewrite the text:
   - In the tone of a National Geographic documentary
   - Narrated as if by Morgan Freeman
3. Request a cinematic, descriptive, slow-paced style.
4. Print the transformed output.



In [21]:
prompt = f"""
Rewrite the following text in the tone of a National Geographic documentary,
as if narrated by Morgan Freeman. Use a slow, cinematic, descriptive style.

Text: ```{text}```
"""

response = get_completion(prompt)
print("=== MORGAN FREEMAN DOCUMENTARY VERSION ===")
print(response)


=== MORGAN FREEMAN DOCUMENTARY VERSION ===
In the early 1900s, a time before the convenience of refrigerators graced our homes, a culinary dilemma plagued households across the land. Food, once fresh and vibrant, succumbed quickly to the relentless march of time. Desperate times called for desperate measures, and thus emerged a solution as ingenious as it was humble.

Enter ketchup, a tomato-based elixir born not of gourmet aspirations, but of sheer necessity. People sought to mask the telltale signs of spoilage in their meals, turning to this tangy concoction to salvage what was salvageable. A symphony of flavors, ketchup danced on the palate, camouflaging the unpleasant tastes that threatened to betray the true nature of the dish.

In a world where freshness was a fleeting luxury, ketchup stood as a stalwart guardian of edibility. Its crimson hue a beacon of hope in a sea of culinary uncertainty, it served as a practical lifeline for those seeking to prolong the lifespan of their pro

## **Conclusion**

### **1. Translation**
The translations into French, Spanish, and Japanese were accurate and preserved the meaning of the original text. The model did not introduce hallucinations or add new historical details. All three outputs maintained a neutral, factual tone.  
**No errors were detected at this stage.**

---

### **2. Spellcheck & Grammar Check**

#### **French**
- **Corrections reported:** 2  
- **Visible corrections:** None  
The corrected text is identical to the original. This suggests the model may have counted invisible adjustments (spacing, punctuation normalization) or overestimated corrections.  
**Observation:** Minor hallucination in the correction count.

#### **Spanish**
- **Corrections reported:** 2  
- **Visible corrections:** 1  
  - “ketchup” → “kétchup” (a stylistic adaptation, not an actual error)  
The rest of the text remained unchanged.  
**Observation:** The model applied a stylistic preference rather than a true correction.

#### **Japanese**
- **Corrections reported:** 4  
- **Visible corrections:** 0  
The corrected text is identical to the original. This indicates the model likely hallucinated corrections due to the complexity of the language.  
**Observation:** Clear mismatch between reported and actual corrections.

---

### **3. Tone Transformation**
The transformation into a National Geographic–style narration, voiced as Morgan Freeman, was highly successful. The output was cinematic, descriptive, and stylistically coherent. The model expanded the narrative but stayed faithful to the original meaning.  
**No errors or hallucinations detected.**

---

### **Overall Learnings**
- The model performs **very well** in translation and tone transformation tasks when prompts are clear and specific.
- Spellchecking across multiple languages reveals limitations:
  - The model may **invent corrections** when unsure (especially in Japanese).
  - It may apply **stylistic changes** instead of true corrections (Spanish).
  - It may report corrections that are not visible (French).
- These behaviors highlight that LLMs are strong in **creative and semantic tasks**, but less reliable in **fine‑grained linguistic accuracy**, particularly in languages with complex grammar systems.